# Programación de eventos y scheduling

## Objetivos de la clase
- Diferenciar planificación vs scheduling
- Comprender métricas: makespan, lateness, throughput
- Implementar algoritmos clásicos
- Conectar con CSP y MILP

## 1. Concepto: Planificación vs Scheduling
- Planificación (Planning)
  - Decide qué hacer y en qué orden
  - Ejemplo
  
- Scheduling
  - Decide cuándo ejecutar cada tarea
  - Incluye:
    - recursos
    - tiempos
    - restricciones

- Relación:
  - Planning → genera tareas
  - Scheduling → las organiza en el tiempo





In [ ]:
# Setup inicial  (sólo ejecutar 1 vez)
!pip install pulp matplotlib numpy pandas

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 3. Definición del problema

Tenemos:
- n tareas
- cada tarea tiene:
  - tiempo de procesamiento p
  - deadline d


In [ ]:
jobs = pd.DataFrame({
    'job': ['A','B','C','D','E'],
    'processing_time': [3, 2, 4, 1, 2],
    'deadline': [5, 3, 7, 4, 6]
})

jobs

# Métricas fundamentales
🔹 Makespan

Tiempo total de finalización:

In [ ]:
def makespan(schedule):
    return sum(job['processing_time'] for job in schedule)

## Lateness
$L_i=C_i−d_i$

In [ ]:
def compute_lateness(schedule):
    time = 0
    lateness = []

    for job in schedule:
        time += job['processing_time']
        L = time - job['deadline']
        lateness.append(L)

    return lateness

## Throughput

Cantidad de tareas completadas

In [ ]:
def throughput(schedule):
    return len(schedule)

# Algoritmos clásicos de Scheduling

In [ ]:
# FCFS (First Come First Serve)

def fcfs(jobs):
    return jobs.to_dict('records')

# SPT (Shortest Processing Time)
## Minimiza tiempo promedio:

def spt(jobs):
    return jobs.sort_values(by='processing_time').to_dict('records')

# EDD (Earliest Due Date)
## Minimiza lateness:
def edd(jobs):
    return jobs.sort_values(by='deadline').to_dict('records')

# Comparamos los algoritmos

In [ ]:
algorithms = {
    "FCFS": fcfs,
    "SPT": spt,
    "EDD": edd
}

results = []

for name, algo in algorithms.items():
    schedule = algo(jobs)

    mk = makespan(schedule)
    lat = compute_lateness(schedule)

    results.append({
        'Algoritmo': name,
        'Makespan': mk,
        'Max Lateness': max(lat)
    })

pd.DataFrame(results)

# Visualización tipo Gantt

In [ ]:
def plot_schedule(schedule, title):
    time = 0
    fig, ax = plt.subplots(figsize=(10,3))

    for job in schedule:
        ax.barh(0, job['processing_time'], left=time)
        ax.text(time + job['processing_time']/2, 0, job['job'],
                ha='center', va='center', color='white')
        time += job['processing_time']

    ax.set_title(title)
    ax.set_yticks([])
    plt.show()

In [ ]:
for name, algo in algorithms.items():
    plot_schedule(algo(jobs), name)

# Scheduling como CSP

Un scheduling puede modelarse como CSP (Constraint Satisfaction Problem):

Variables:

- start_time[i]

Restricciones:

- no superposición:

$start_i+p_i ≤ start_j$ o viceversa

Ejemplo simple:

In [ ]:
from itertools import permutations

def is_valid(order):
    return True  # simplificado

best = None

for perm in permutations(jobs.to_dict('records')):
    if is_valid(perm):
        best = perm
        break

best

# Scheduling como MILP

Usamos PuLP

In [ ]:
from pulp import *

def milp_schedule(jobs):
    model = LpProblem("Scheduling", LpMinimize)

    start = {i: LpVariable(f"start_{i}", lowBound=0) for i in jobs.index}

    # Makespan
    Cmax = LpVariable("Cmax", lowBound=0)

    # Objetivo
    model += Cmax

    # Restricciones
    for i in jobs.index:
        model += start[i] + jobs.loc[i,'processing_time'] <= Cmax

    # No solapamiento (simplificado)
    # (esto normalmente requiere variables binarias)

    model.solve()

    return {i: value(start[i]) for i in jobs.index}

In [ ]:
from pulp import *
import pandas as pd

def milp_schedule(jobs):
    n = len(jobs)
    M = 1000  # suficientemente grande

    model = LpProblem("Scheduling", LpMinimize)

    # Variables
    s = LpVariable.dicts("start", range(n), lowBound=0)
    Cmax = LpVariable("Cmax", lowBound=0)

    # Variables binarias
    x = {}
    for i in range(n):
        for j in range(n):
            if i != j:
                x[(i,j)] = LpVariable(f"x_{i}_{j}", cat='Binary')

    # Objetivo
    model += Cmax

    # Makespan
    for i in range(n):
        model += s[i] + jobs.loc[i,'processing_time'] <= Cmax

    # No solapamiento
    for i in range(n):
        for j in range(n):
            if i < j:
                pi = jobs.loc[i,'processing_time']
                pj = jobs.loc[j,'processing_time']

                model += s[i] + pi <= s[j] + M*(1 - x[(i,j)])
                model += s[j] + pj <= s[i] + M*(x[(i,j)])

    # Resolver
    model.solve()

    schedule = []
    for i in range(n):
        schedule.append({
            'job': jobs.loc[i,'job'],
            'start': value(s[i]),
            'duration': jobs.loc[i,'processing_time']
        })

    return pd.DataFrame(schedule).sort_values(by='start')

In [ ]:
schedule = milp_schedule(jobs)
schedule

In [ ]:
def plot_milp(schedule):
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(10,3))

    for _, row in schedule.iterrows():
        ax.barh(0, row['duration'], left=row['start'])
        ax.text(row['start'] + row['duration']/2, 0, row['job'],
                ha='center', va='center', color='white')

    ax.set_title("MILP Scheduling")
    ax.set_yticks([])
    plt.show()

plot_milp(schedule)

## Problemas del enfoque MILP

- Escala mal: $O(n^2)$ variables binarias
- Sensible a:
  - elección de $M$
  - simetrías

## Mejoras posibles
- restricciones de precedencia
- reducción de simetría
- formulations tipo:
  - time-indexed
  - disjunctive programming

# Conexión con CSP

Este modelo es equivalente a:

- CSP con:
  - variables: $s_i$
  - restricciones disyuntivas

Pero MILP permite:
- optimizar directamente (no solo factibilidad)